# LSTM Next Word Prediction Model

This notebook implements a deep learning model using LSTM (Long Short-Term Memory) networks to predict the next word in a sequence.

## What you'll learn:
- How to preprocess text data for sequence modeling
- How to build an LSTM model for text prediction
- How to train and evaluate the model
- How to generate predictions with temperature sampling

## 1. Install Dependencies

In [ ]:
!pip install tensorflow numpy matplotlib -q

## 2. Import Libraries

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 3. Prepare Training Data

We'll create a sample text corpus for training. In a real project, you would use a larger dataset like books, articles, or Wikipedia pages.

In [ ]:
# Sample training text - you can replace this with your own dataset
training_text = """
the quick brown fox jumps over the lazy dog
the quick brown fox jumps over the lazy cat
a quick brown fox jumps over a lazy dog
the lazy dog sleeps all day long
the lazy cat sleeps all night long
a lazy dog is a happy dog to have
the quick brown cat runs very fast
the quick brown dog runs very fast
a quick brown cat is a happy cat indeed
deep learning is a subset of machine learning
machine learning is a subset of artificial intelligence
artificial intelligence is the future of technology
natural language processing is an important field of study
neural networks are powerful tools for deep learning tasks
recurrent neural networks are used for sequential data processing
long short term memory networks solve the vanishing gradient problem effectively
transformers have revolutionized natural language processing tasks recently
attention mechanisms allow models to focus on relevant parts of input
python is a popular programming language for machine learning projects
tensorflow and pytorch are popular deep learning frameworks today
the field of artificial intelligence continues to grow rapidly
deep learning models require large amounts of data to train effectively
natural language processing enables computers to understand human language
sequence to sequence models are used for translation and summarization
word embeddings capture semantic relationships between words in text
the training process involves optimizing the model weights iteratively
gradient descent is a common optimization algorithm used in deep learning
the loss function measures how well the model predictions match the targets
batch normalization helps stabilize and accelerate the training process
dropout regularization prevents overfitting in neural network models
transfer learning allows reusing pre trained models for new tasks
generative models can create new content such as text images and music
reinforcement learning involves agents learning through trial and error
computer vision enables machines to interpret and understand visual data
speech recognition converts spoken language into written text accurately
sentiment analysis determines the emotional tone of a piece of text
named entity recognition identifies important entities in text data
text classification assigns predefined categories to pieces of text
language models predict the probability of sequences of words in text
the attention mechanism was introduced to improve sequence modeling tasks
bert and gpt are popular pre trained language models used today
""".strip()

print(f"Training text length: {len(training_text)} characters")
print(f"Sample text: {training_text[:200]}...")

## 4. Text Preprocessing

We need to:
1. Tokenize the text (convert words to numbers)
2. Create input sequences (n-grams)
3. Pad sequences to fixed length
4. Create input (X) and output (y) pairs

In [ ]:
# Hyperparameters
MAX_SEQUENCE_LENGTH = 50  # Maximum length of input sequences
EMBEDDING_DIM = 64        # Dimension of word embeddings
LSTM_UNITS = 128          # Number of LSTM units

# Initialize tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts([training_text])
total_words = len(tokenizer.word_index) + 1

print(f"Vocabulary size: {total_words}")
print(f"\nFirst 20 word indices: {dict(list(tokenizer.word_index.items())[:20])}")

In [ ]:
# Create input sequences using n-grams
input_sequences = []

for line in training_text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    # Create n-gram sequences
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences
max_sequence_len = min(MAX_SEQUENCE_LENGTH, max([len(x) for x in input_sequences]))
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

print(f"Total sequences: {len(input_sequences)}")
print(f"Sequence length: {max_sequence_len}")
print(f"\nExample sequence: {input_sequences[0]}")

In [ ]:
# Create predictors (X) and label (y)
X = input_sequences[:, :-1]  # All columns except the last
y = input_sequences[:, -1]   # Only the last column

# One-hot encode the output
y = to_categorical(y, num_classes=total_words)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nExample X[0]: {X[0]}")
print(f"Example y[0] (one-hot): {y[0][:10]}... (showing first 10 values)")

## 5. Build the LSTM Model

The model consists of:
- **Embedding Layer**: Converts word indices to dense vectors
- **LSTM Layers**: Capture sequential dependencies
- **Dense Layers**: Process features and output predictions
- **Dropout**: Prevents overfitting

In [ ]:
def build_model(vocab_size, embedding_dim, lstm_units, max_sequence_len):
    """
    Build an LSTM model for next word prediction.
    
    Args:
        vocab_size: Size of the vocabulary
        embedding_dim: Dimension of word embeddings
        lstm_units: Number of LSTM units
        max_sequence_len: Length of input sequences
    
    Returns:
        Compiled Keras model
    """
    model = Sequential([
        # Embedding layer: converts word indices to dense vectors
        Embedding(
            input_dim=vocab_size,
            output_dim=embedding_dim,
            input_length=max_sequence_len - 1,
            name='embedding'
        ),
        
        # First LSTM layer with return sequences for stacking
        LSTM(
            units=lstm_units,
            return_sequences=True,
            dropout=0.2,
            recurrent_dropout=0.2,
            name='lstm_1'
        ),
        
        # Second LSTM layer
        LSTM(
            units=lstm_units,
            dropout=0.2,
            recurrent_dropout=0.2,
            name='lstm_2'
        ),
        
        # Dense hidden layer
        Dense(units=lstm_units, activation='relu', name='dense_hidden'),
        Dropout(0.2, name='dropout'),
        
        # Output layer with softmax activation
        Dense(units=vocab_size, activation='softmax', name='output')
    ])
    
    # Compile model
    model.compile(
        loss='categorical_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )
    
    return model

# Build the model
model = build_model(total_words, EMBEDDING_DIM, LSTM_UNITS, max_sequence_len)

# Display model summary
model.summary()

## 6. Train the Model

We'll train the model with:
- **Early Stopping**: Stops training when loss stops improving
- **Validation Split**: Uses 10% of data for validation

In [ ]:
# Training hyperparameters
EPOCHS = 100
BATCH_SIZE = 16
VALIDATION_SPLIT = 0.1

# Callbacks
callbacks = [
    EarlyStopping(
        monitor='loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    )
]

# Train the model
print("Starting training...")
print("=" * 50)

history = model.fit(
    X, y,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    callbacks=callbacks,
    verbose=1
)

print("=" * 50)
print("Training complete!")

## 7. Visualize Training Results

Let's plot the loss and accuracy curves to understand model performance.

In [ ]:
def plot_training_history(history):
    """
    Plot training and validation loss/accuracy.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot Loss
    axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
    axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    axes[0].set_title('Model Loss', fontsize=14)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot Accuracy
    axes[1].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
    axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
    axes[1].set_title('Model Accuracy', fontsize=14)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_training_history(history)

## 8. Make Predictions

Now let's use our trained model to predict the next word(s) given a seed text.

In [ ]:
def predict_next_word(seed_text, model, tokenizer, max_sequence_len, temperature=1.0):
    """
    Predict the next word given a seed text.
    
    Args:
        seed_text: Initial text to start prediction
        model: Trained Keras model
        tokenizer: Fitted tokenizer
        max_sequence_len: Maximum sequence length
        temperature: Sampling temperature (lower = more conservative)
    
    Returns:
        Predicted word
    """
    # Tokenize and pad the input
    token_list = tokenizer.texts_to_sequences([seed_text.lower()])[0]
    token_list = pad_sequences([token_list], maxlen=max_sequence_len - 1, padding='pre')
    
    # Get prediction probabilities
    predicted_probs = model.predict(token_list, verbose=0)[0]
    
    # Apply temperature scaling
    if temperature != 1.0:
        predicted_probs = np.log(predicted_probs + 1e-8) / temperature
        predicted_probs = np.exp(predicted_probs)
        predicted_probs = predicted_probs / np.sum(predicted_probs)
    
    # Sample from the distribution
    predicted_index = np.random.choice(len(predicted_probs), p=predicted_probs)
    
    # Convert index to word
    for word, index in tokenizer.word_index.items():
        if index == predicted_index:
            return word
    return ''


def generate_text(seed_text, num_words, model, tokenizer, max_sequence_len, temperature=1.0):
    """
    Generate multiple words given a seed text.
    
    Args:
        seed_text: Initial text
        num_words: Number of words to generate
        model: Trained Keras model
        tokenizer: Fitted tokenizer
        max_sequence_len: Maximum sequence length
        temperature: Sampling temperature
    
    Returns:
        Generated text
    """
    result = seed_text.lower()
    
    for _ in range(num_words):
        next_word = predict_next_word(result, model, tokenizer, max_sequence_len, temperature)
        if next_word:
            result += ' ' + next_word
        else:
            break
    
    return result


def get_top_predictions(seed_text, model, tokenizer, max_sequence_len, top_k=5):
    """
    Get top-k predictions for the next word.
    
    Args:
        seed_text: Input text
        model: Trained Keras model
        tokenizer: Fitted tokenizer
        max_sequence_len: Maximum sequence length
        top_k: Number of top predictions to return
    
    Returns:
        List of (word, probability) tuples
    """
    # Tokenize and pad
    token_list = tokenizer.texts_to_sequences([seed_text.lower()])[0]
    token_list = pad_sequences([token_list], maxlen=max_sequence_len - 1, padding='pre')
    
    # Get predictions
    predicted_probs = model.predict(token_list, verbose=0)[0]
    
    # Get top-k indices
    top_indices = np.argsort(predicted_probs)[-top_k:][::-1]
    
    # Convert to words
    top_predictions = []
    for idx in top_indices:
        for word, index in tokenizer.word_index.items():
            if index == idx:
                top_predictions.append((word, float(predicted_probs[idx])))
                break
    
    return top_predictions

print("Prediction functions defined successfully!")

## 9. Test the Model

Let's test our model with various seed texts and see the predictions.

In [ ]:
# Test phrases
test_phrases = [
    "the quick",
    "deep learning",
    "natural language",
    "neural networks are",
    "python is a",
    "artificial intelligence",
    "machine learning is"
]

print("=" * 60)
print("NEXT WORD PREDICTIONS")
print("=" * 60)

for phrase in test_phrases:
    print(f"\nSeed: '{phrase}'")
    print("-" * 40)
    
    # Get top predictions
    top_preds = get_top_predictions(phrase, model, tokenizer, max_sequence_len, top_k=5)
    print("Top 5 predictions:")
    for i, (word, prob) in enumerate(top_preds, 1):
        print(f"  {i}. {word}: {prob:.4f}")
    
    # Generate completion
    completion = generate_text(phrase, 5, model, tokenizer, max_sequence_len, temperature=0.7)
    print(f"Generated: '{completion}'")

## 10. Temperature Sampling

Temperature controls the randomness of predictions:
- **Low temperature (0.1-0.5)**: More conservative, sticks to likely words
- **Medium temperature (0.7-1.0)**: Balanced creativity
- **High temperature (1.5-2.0)**: More random, creative outputs

In [ ]:
# Compare different temperatures
seed_text = "deep learning"
temperatures = [0.2, 0.5, 0.7, 1.0, 1.5]

print("=" * 60)
print(f"Temperature Comparison for seed: '{seed_text}'")
print("=" * 60)

for temp in temperatures:
    result = generate_text(seed_text, 8, model, tokenizer, max_sequence_len, temperature=temp)
    print(f"\nTemperature {temp}: {result}")

## 11. Interactive Prediction

Try your own seed text below!

In [ ]:
# Interactive prediction - modify this text to try different inputs
YOUR_SEED_TEXT = "the lazy dog"  # <-- Change this!
NUM_WORDS_TO_GENERATE = 10        # <-- Change this!
TEMPERATURE = 0.7                 # <-- Change this (0.1 to 2.0)!

# Generate text
generated = generate_text(
    YOUR_SEED_TEXT,
    NUM_WORDS_TO_GENERATE,
    model,
    tokenizer,
    max_sequence_len,
    temperature=TEMPERATURE
)

print(f"Input: '{YOUR_SEED_TEXT}'")
print(f"Generated ({NUM_WORDS_TO_GENERATE} words, temp={TEMPERATURE}): '{generated}'")

## 12. Save the Model

Save your trained model for later use.

In [ ]:
import pickle

# Save the model
model.save('lstm_next_word_model.keras')
print("Model saved to 'lstm_next_word_model.keras'")

# Save the tokenizer
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
print("Tokenizer saved to 'tokenizer.pkl'")

# Save model config
config = {
    'max_sequence_len': max_sequence_len,
    'vocab_size': total_words,
    'embedding_dim': EMBEDDING_DIM,
    'lstm_units': LSTM_UNITS
}
with open('model_config.pkl', 'wb') as f:
    pickle.dump(config, f)
print("Model config saved to 'model_config.pkl'")

## Summary

In this notebook, we:

1. **Preprocessed text data** by tokenizing and creating n-gram sequences
2. **Built an LSTM model** with embedding layers, two LSTM layers, and dense layers
3. **Trained the model** using categorical crossentropy loss and Adam optimizer
4. **Generated predictions** using temperature sampling to control creativity
5. **Saved the model** for future use

### Next Steps:
- Use a larger dataset (e.g., books, Wikipedia articles)
- Experiment with different model architectures (GRU, Bidirectional LSTM)
- Add attention mechanisms
- Use pre-trained word embeddings (GloVe, Word2Vec)
- Try Transformer-based models for better performance